In [10]:
import os
os.listdir()

['pge_style_data_dictionary.csv',
 'pge_style_service_requests.csv',
 'pge_style_practice.db',
 'pge_march_prep_day1.ipynb',
 '.ipynb_checkpoints']

In [2]:
import pandas as pd
df = pd.read_csv("pge_style_service_requests.csv")

In [3]:
# Quick look
df.head()

,request_id,customer_id,region_id,request_type,priority,channel,status,created_at,closed_at,resolution_hours,crew_id,satisfaction_score,repeat_contact,estimated_cost
0,1,1606,7,Vegetation,Critical,Web,Closed,2024-01-27 19:28:00,2024-01-29 05:28:00,34.0,20.0,4.0,0,408.29
1,2,521,2,Inspection,Medium,Field Report,Closed,2025-05-21 19:34:00,2025-05-27 13:34:00,138.0,139.0,4.0,0,328.93
2,3,5109,4,Vegetation,Medium,Field Report,In Progress,2026-01-05 04:53:00,NaN,NaN,82.0,NaN,0,419.07
3,4,1607,5,Billing,High,Mobile App,Closed,2025-11-28 16:00:00,2025-11-29 23:00:00,31.0,NaN,4.0,0,78.29
4,5,5122,1,Line Repair,Medium,Mobile App,In Progress,2024-01-31 21:57:00,NaN,NaN,100.0,NaN,0,371.78


In [4]:
# Info + data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30100 entries, 0 to 30099
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   request_id          30100 non-null  int64  
 1   customer_id         30100 non-null  int64  
 2   region_id           30100 non-null  int64  
 3   request_type        30100 non-null  object 
 4   priority            29804 non-null  object 
 5   channel             30100 non-null  object 
 6   status              30100 non-null  object 
 7   created_at          30100 non-null  object 
 8   closed_at           20963 non-null  object 
 9   resolution_hours    20963 non-null  float64
 10  crew_id             24722 non-null  float64
 11  satisfaction_score  20963 non-null  float64
 12  repeat_contact      30100 non-null  int64  
 13  estimated_cost      30100 non-null  float64
dtypes: float64(4), int64(4), object(6)
memory usage: 3.2+ MB


In [5]:
import pandas as pd

df = pd.read_csv("pge_style_service_requests.csv")
df.shape

(30100, 14)

In [6]:
# missing values
missing = df.isna().sum().sort_values(ascending=False)
missing

closed_at             9137
resolution_hours      9137
satisfaction_score    9137
crew_id               5378
priority               296
request_id               0
channel                  0
request_type             0
region_id                0
customer_id              0
status                   0
created_at               0
repeat_contact           0
estimated_cost           0
dtype: int64

In [7]:
missing[missing > 0]

closed_at             9137
resolution_hours      9137
satisfaction_score    9137
crew_id               5378
priority               296
dtype: int64

In [8]:
# most common request types
top_types = (df.groupby("request_type")
               .size()
               .sort_values(ascending=False))

In [9]:
top_types.head(5)

request_type
Outage         6125
Billing        5378
Meter          4589
Line Repair    4400
Inspection     3590
dtype: int64

### Step 1 Insight

Here's the key interpretation:

* 'closed_at', 'resolution_hours', 'satisfaction_score' are missing mostly because those requests are **not closed yet**.

* 'crew_id' is missing because not every request needs a crew (ex:billing).

* 'priority' has missing values (296) = **true data quality issue** worth noting.

This dataset has 30,100 service request records across 14 fields, including request type, status, timing, and cost. Missing values are mostly expected for requests (e.g 'closed_at, 'resolution_hours', 'satisfaction_score') and for request types that don't require crews, while the missing 'priority' values look like data quality gap to address before deeper analysis; the highest-volume request types are Outage, Billing, and Meter


In [11]:
df["status"].value_counts()

status
Closed         20963
In Progress     3716
Pending         2411
Open            1807
Escalated       1203
Name: count, dtype: int64

In [12]:
df[df["closed_at"].isna()]["status"].value_counts()

status
In Progress    3716
Pending        2411
Open           1807
Escalated      1203
Name: count, dtype: int64